# 12 — Candidate exceptions, viability, and qualitative case-finding (2024)

This notebook answers the part of Workplan Question 3 that a regression table
cannot answer by itself:

> If there are candidates who do not follow the general finance-to-support
> pattern, who are they, and what is different about them?

Notebooks 10 and 11 described the **general** pattern. This notebook is about
the **exceptions** — and exceptions are usually where the interesting politics
lives.

Two tools:

1. **Residuals** — actual support minus the support predicted from money.
2. **Viability comparisons** — do viable candidates have larger finance
   profiles, and who are the exceptions?

The output is a **shortlist of candidates for qualitative research**, not a
claim that the residual explains anything. The regression finds the case; bios,
campaign history, endorsements, and coalitions explain it.


## 1. Setup and analysis table

We rebuild the same table as notebook 10, with the same handling of candidates
who filed spending through more than one committee, so the three Question 3
notebooks are directly comparable.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

# ------------------------------------------------------------
# Find the repository root, so this notebook runs from either
# the repo root or the notebooks/ folder.
# ------------------------------------------------------------

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find the repository root. "
        "Expected pyproject.toml in the current directory or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


In [2]:
def collapse_spending_to_candidate(spending):
    """Combine multiple ORESTAR spending rows for the same candidate.

    Upstream, spending is summarised per *filing source* (profile_key), so a
    candidate who filed through more than one committee appears more than once.
    We add those rows together, because a candidate's total spending is the sum
    across all of their filings.

    Averages are recomputed from the combined totals rather than averaged, since
    averaging two averages would weight a small filing the same as a large one.
    """
    spending = spending[spending["candidate_key"].notna()].copy()

    # Report which candidates had more than one filing, so this is visible
    # rather than happening quietly.
    counts = spending.groupby("candidate_key").size()
    multi_filers = counts[counts > 1]

    if len(multi_filers) > 0:
        print(f"Candidates with more than one spending filing: {len(multi_filers)}")
        columns_to_show = ["district", "canonical_candidate", "total_spending", "expenditure_count"]
        columns_to_show = [c for c in columns_to_show if c in spending.columns]
        display(
            spending[spending["candidate_key"].isin(multi_filers.index)]
            .sort_values("candidate_key")[["candidate_key"] + columns_to_show]
        )
    else:
        print("No candidate had more than one spending filing.")

    # Add up the totals per candidate.
    combined = spending.groupby("candidate_key", as_index=False).agg(
        total_spending=("total_spending", "sum"),
        expenditure_count=("expenditure_count", "sum"),
        spending_filings=("candidate_key", "size"),
    )

    # Recompute the average from the combined totals.
    combined["average_expenditure"] = (
        combined["total_spending"] / combined["expenditure_count"]
    )

    return combined


In [3]:
support_path = PROCESSED / "ballot_support" / str(YEAR) / "candidate_ballot_support_2024.csv"
fundraising_path = fundraising_processed_dir(YEAR, CONTEST) / "openelections_candidate_fundraising_summary.csv"
spending_path = spending_processed_dir(YEAR, CONTEST) / "orestar_candidate_spending_summary.csv"

support = pd.read_csv(support_path, low_memory=False)
fundraising = pd.read_csv(fundraising_path, low_memory=False)
spending_raw = pd.read_csv(spending_path, low_memory=False)

# Combine multi-filing candidates (see the function above).
spending = collapse_spending_to_candidate(spending_raw)

# Fundraising should already be one row per candidate. Check rather than assume.
if fundraising["candidate_key"].duplicated().any():
    raise ValueError(
        "Duplicate candidate_key values in the fundraising summary. "
        "Inspect the fundraising notebook before continuing."
    )

# Start from ballot support and attach finance with LEFT joins, so a candidate
# never disappears just because a finance record is missing.
analysis = support.merge(
    fundraising[[
        "candidate_key",
        "total_amount",
        "total_contribution_count",
        "average_contribution",
        "median_contribution",
    ]],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

analysis = analysis.merge(
    spending[[
        "candidate_key",
        "total_spending",
        "expenditure_count",
        "average_expenditure",
        "spending_filings",
    ]],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

# Friendlier names for the two headline finance columns.
analysis = analysis.rename(columns={
    "total_amount": "fundraising",
    "total_contribution_count": "contribution_count",
})

# 0/1 version of viability, for correlations and regressions later.
analysis["viable_int"] = analysis["is_viable"].astype(bool).astype(int)

# Log dollars. log1p(x) is log(1 + x), which is also defined at x = 0.
analysis["log_fundraising"] = np.log1p(analysis["fundraising"])
analysis["log_spending"] = np.log1p(analysis["total_spending"])

print()
print("Candidates on ballot:  ", len(analysis))
print("Missing fundraising:   ", analysis["fundraising"].isna().sum())
print("Missing spending:      ", analysis["total_spending"].isna().sum())


No candidate had more than one spending filing.

Candidates on ballot:   98
Missing fundraising:    33
Missing spending:       33


## 2. What is a residual?

The regression from notebook 10 makes a prediction for every candidate: given
how much money you raised, this is roughly the support we would expect.

The **residual** is the gap between what actually happened and that prediction:

`residual = actual support − predicted support`

Suppose the model predicts a candidate should receive 9,000 ballot mentions:

- if they actually received 14,000, the residual is **+5,000** — they
  **over-performed** their money;
- if they actually received 6,000, the residual is **−3,000** — they
  **under-performed** it.

**We fit a separate model inside each district.** This matters for Portland:
each district ran its own contest with its own number of ballots, so a candidate
should be compared against the people who were actually on the ballot with them,
not against the whole city. A district with more voters would otherwise make
every candidate look like an over-performer.

The residual tells us **where to look**, never **why**.


In [4]:
def district_residuals(data, money, outcome, money_label):
    """Fit one regression per district and return every candidate's residual.

    Using log money matches notebook 10, where the log specification fit best.
    """
    all_districts = []

    districts = sorted(data["district"].dropna().unique())

    for district in districts:
        columns_needed = [
            "candidate_key",
            "canonical_candidate",
            "district",
            "is_viable",
            money,
            outcome,
        ]

        district_data = data.loc[
            data["district"] == district, columns_needed
        ].dropna().copy()

        # Skip districts with too few candidates to fit a line.
        if len(district_data) < 4 or district_data[money].nunique() < 2:
            continue

        x = np.log1p(district_data[money].to_numpy(dtype=float))
        y = district_data[outcome].to_numpy(dtype=float)

        model = sm.OLS(y, sm.add_constant(x)).fit()

        district_data["predicted"] = model.predict(sm.add_constant(x))
        district_data["residual"] = district_data[outcome] - district_data["predicted"]

        # Record which model produced these residuals.
        district_data["money_measure"] = money_label
        district_data["outcome_measure"] = outcome
        district_data["district_r_squared"] = model.rsquared

        all_districts.append(district_data)

    return pd.concat(all_districts, ignore_index=True)


# Build residuals for both money measures and both outcomes.
residual_tables = []

for money, money_label in [("fundraising", "Fundraising"), ("total_spending", "Spending")]:
    for outcome in ["mentions", "first_place_votes"]:
        residual_tables.append(
            district_residuals(analysis, money, outcome, money_label)
        )

residuals = pd.concat(residual_tables, ignore_index=True)

print("Residual rows (candidates x money measures x outcomes):", len(residuals))
display(residuals.head())


Residual rows (candidates x money measures x outcomes): 260


,candidate_key,canonical_candidate,district,is_viable,fundraising,mentions,predicted,residual,money_measure,outcome_measure,district_r_squared,first_place_votes,total_spending
0,2024|1|timur ender,Timur Ender,1,True,61809.84,16858.0,19069.020383,-2211.020383,Fundraising,mentions,0.93281,NaN,NaN
1,2024|1|terrence hayes,Terrence Hayes,1,True,36108.05,15967.0,17108.633258,-1141.633258,Fundraising,mentions,0.93281,NaN,NaN
2,2024|1|loretta smith,Loretta Smith,1,True,43744.96,17984.0,17808.329766,175.670234,Fundraising,mentions,0.93281,NaN,NaN
3,2024|1|deian salazar,Deian Salazar,1,False,1380.00,5262.0,5205.812703,56.187297,Fundraising,mentions,0.93281,NaN,NaN
4,2024|1|noah ernst,Noah Ernst,1,True,12993.18,11606.0,13381.253542,-1775.253542,Fundraising,mentions,0.93281,NaN,NaN


## 3. Who over- and under-performs their fundraising?

We start with **fundraising and ballot mentions**, because notebook 10 suggested
money tracks broad ballot consideration more closely than first-place support.

Three from each district in each direction, so no single district dominates the
list.


In [5]:
# Keep just the fundraising-and-mentions residuals.
fundraising_mentions = residuals[
    (residuals["money_measure"] == "Fundraising")
    & (residuals["outcome_measure"] == "mentions")
].copy()

columns_to_show = [
    "district",
    "canonical_candidate",
    "is_viable",
    "fundraising",
    "mentions",
    "predicted",
    "residual",
]

overperformers = (
    fundraising_mentions
    .sort_values("residual", ascending=False)
    .groupby("district")
    .head(3)
)

underperformers = (
    fundraising_mentions
    .sort_values("residual", ascending=True)
    .groupby("district")
    .head(3)
)

print("=== OVER-performed: more mentions than their money predicted ===")
display(overperformers[columns_to_show].round(0))

print("=== UNDER-performed: fewer mentions than their money predicted ===")
display(underperformers[columns_to_show].round(0))


=== OVER-performed: more mentions than their money predicted ===


,district,canonical_candidate,is_viable,fundraising,mentions,predicted,residual
62,4,Olivia Clark,True,95639.0,45205.0,26714.0,18491.0
45,3,Matthew (Matt) Anderson,False,710.0,4506.0,-11083.0,15589.0
56,4,Eric Zimmerman,True,41879.0,36731.0,22981.0,13750.0
31,2,Michelle DePass,True,32578.0,32612.0,19557.0,13055.0
19,2,Elana Pirtle-Guiney,True,47040.0,34268.0,22362.0,11906.0
57,4,Sarah Silkie,True,25478.0,32039.0,20735.0,11304.0
25,2,Sameer Kanal,True,34712.0,31168.0,20042.0,11126.0
35,3,Steve Novick,True,87241.0,51189.0,40295.0,10894.0
33,3,Angelita Morillo,True,64641.0,47477.0,37092.0,10385.0
11,1,Candace Avalos,True,58894.0,22267.0,18893.0,3374.0


=== UNDER-performed: fewer mentions than their money predicted ===


,district,canonical_candidate,is_viable,fundraising,mentions,predicted,residual
50,4,Moses Ross,False,27897.0,5238.0,21145.0,-15907.0
53,4,Stanley Penkin,False,61847.0,9570.0,24744.0,-15174.0
36,3,Harrison Kass,False,16058.0,9223.0,22217.0,-12994.0
49,4,Mike DiNapoli,False,7289.0,3001.0,15078.0,-12077.0
27,2,Nabil Zaghloul,False,26484.0,6482.0,17976.0,-11494.0
15,2,Debbie Kitchin,False,34885.0,9088.0,20080.0,-10992.0
43,3,Theo Hathaway Saner,False,7505.0,3525.0,14092.0,-10567.0
37,3,Luke Zak,False,9731.0,8581.0,16866.0,-8285.0
21,2,Michael (Mike) Marshall,False,55075.0,15684.0,23565.0,-7881.0
8,1,Cayle Tern,False,7395.0,9061.0,11326.0,-2265.0


## 4. Interactive residual explorer

A bar chart reads far better than a scatter for this question: the dashed line
at zero is exactly what the model expected, so distance from it is the whole
story.

- bars to the **right**: did better than money predicted;
- bars to the **left**: did worse than money predicted.

**Edit the three settings and rerun** to switch district, money measure, or
outcome. Comparing fundraising residuals against spending residuals for the same
district is especially useful: a candidate who over-performs on both is a
stronger case than one who only appears in a single view.


In [6]:
# ------------------------------------------------------------
# EDIT THESE THREE SETTINGS.
# money_to_explore:   "Fundraising" or "Spending"
# outcome_to_explore: "mentions" or "first_place_votes"
# ------------------------------------------------------------
district_to_explore = 3
money_to_explore = "Fundraising"
outcome_to_explore = "mentions"
# ------------------------------------------------------------

case_data = residuals[
    (residuals["district"] == district_to_explore)
    & (residuals["money_measure"] == money_to_explore)
    & (residuals["outcome_measure"] == outcome_to_explore)
].copy()

case_data = case_data.sort_values("residual")

fig = px.bar(
    case_data,
    x="residual",
    y="canonical_candidate",
    color="is_viable",
    orientation="h",
    hover_data={
        "predicted": ":,.0f",
        outcome_to_explore: ":,.0f",
        "residual": ":,.0f",
        "district_r_squared": ":.3f",
    },
    labels={
        "residual": "Actual minus predicted support",
        "canonical_candidate": "Candidate",
        "is_viable": "Viable",
    },
    title=(
        f"District {district_to_explore} - {money_to_explore} residuals "
        f"for {outcome_to_explore}"
    ),
)

# The zero line is the model's expectation.
fig.add_vline(x=0, line_dash="dash")
fig.show()


### How to turn an outlier into a qualitative question

This is the most important paragraph in the notebook.

Do **not** write:

> Candidate X over-performed because of incumbency.

The residual contains no information about incumbency. It only says the
finance-only model was wrong for this candidate. Write instead:

> Candidate X received substantially more support than the finance-only model
> predicted. We should investigate whether prior office, name recognition,
> endorsements, organised constituency support, or slate effects help explain
> the gap.

The regression **identifies** the case. Bio and context research
**investigates** it. Keeping those two steps separate is what makes the
qualitative section of the report defensible rather than speculative.

One more caution: a large residual can also mean the **data** are wrong for that
candidate — a missing filing, or independent spending mixed into their total. Rule
that out before reaching for a political explanation.


## 5. Viability: useful, but not an independent test

Our viability measure is defined from ballot mentions relative to each
district's STV threshold. So `mentions` and `is_viable` are **built from the same
underlying quantity**.

This means finance-to-mentions and finance-to-viability are two views of one
relationship, not two independent confirmations of it. If both look strong, that
is **one** finding, and the report should say so rather than presenting them as
mutually reinforcing evidence.

Viability is still worth examining, because it turns a continuous pattern into
the question people actually care about: which campaigns crossed the line into
being competitive?


In [7]:
viability_summary = analysis.groupby("is_viable", as_index=False).agg(
    candidates=("candidate_key", "size"),
    median_fundraising=("fundraising", "median"),
    mean_fundraising=("fundraising", "mean"),
    median_contribution_count=("contribution_count", "median"),
    median_spending=("total_spending", "median"),
    mean_spending=("total_spending", "mean"),
    median_expenditure_count=("expenditure_count", "median"),
)

display(viability_summary.round(0))


,is_viable,candidates,median_fundraising,mean_fundraising,median_contribution_count,median_spending,mean_spending,median_expenditure_count
0,False,69,7281.0,13988.0,276.0,16523.0,36051.0,78.0
1,True,29,49453.0,53902.0,1055.0,136928.0,233219.0,192.0


**Why compare medians, not just means.** The mean is pulled around by a single
very large campaign, while the median describes a typical candidate. If the two
differ a lot within the same group, that itself says the group contains an
extreme case worth identifying.


## 6. Interactive viability distributions

A box plot with every candidate shown as a point. The box covers the middle half
of each group, and the line inside it is the median.

What to look for: **overlap**. If the two boxes barely overlap, money separates
viable from non-viable candidates cleanly. If they overlap a lot, plenty of
candidates raised viable-level money and still fell short — and those are exactly
the cases for the shortlist below.


In [8]:
plot_data = analysis.dropna(subset=["fundraising"]).copy()

fig = px.box(
    plot_data,
    x="is_viable",
    y="fundraising",
    points="all",
    hover_name="canonical_candidate",
    hover_data=["district", "mentions", "first_place_votes"],
    log_y=True,
    labels={
        "is_viable": "Viable",
        "fundraising": "Fundraising ($, log scale)",
    },
    title="Fundraising: viable vs. non-viable candidates",
)

fig.show()


In [9]:
plot_data = analysis.dropna(subset=["total_spending"]).copy()

fig = px.box(
    plot_data,
    x="is_viable",
    y="total_spending",
    points="all",
    hover_name="canonical_candidate",
    hover_data=["district", "mentions", "first_place_votes"],
    log_y=True,
    labels={
        "is_viable": "Viable",
        "total_spending": "Reported spending ($, log scale)",
    },
    title="Spending: viable vs. non-viable candidates",
)

fig.show()


## 7. Correlations with viability

Pearson correlation between a continuous variable and a 0/1 variable has its own
name — the **point-biserial correlation** — but nothing different is being
calculated. It is the ordinary correlation formula applied to a yes/no outcome.

We compute it pooled and per district, to see whether money separates viable
from non-viable candidates equally well everywhere.


In [10]:
finance_measures = ["fundraising", "total_spending", "contribution_count", "expenditure_count"]

viability_rows = []

for money in finance_measures:
    if money not in analysis.columns:
        continue

    # Pooled across all districts.
    pair = analysis[["viable_int", money]].dropna()

    if len(pair) >= 3 and pair["viable_int"].nunique() == 2:
        viability_rows.append({
            "scope": "all districts",
            "district": np.nan,
            "finance_measure": money,
            "n": len(pair),
            "correlation": pair["viable_int"].corr(pair[money]),
        })

    # Then one row per district.
    for district in sorted(analysis["district"].dropna().unique()):
        pair = analysis.loc[
            analysis["district"] == district, ["viable_int", money]
        ].dropna()

        # Needs both viable and non-viable candidates to compute anything.
        if len(pair) >= 3 and pair["viable_int"].nunique() == 2:
            viability_rows.append({
                "scope": "district",
                "district": district,
                "finance_measure": money,
                "n": len(pair),
                "correlation": pair["viable_int"].corr(pair[money]),
            })

viability_correlations = pd.DataFrame(viability_rows)
display(viability_correlations.round(3))


,scope,district,finance_measure,n,correlation
0,all districts,NaN,fundraising,65,0.705
1,district,1.0,fundraising,13,0.835
2,district,2.0,fundraising,20,0.669
3,district,3.0,fundraising,15,0.919
4,district,4.0,fundraising,17,0.506
5,all districts,NaN,total_spending,65,0.438
6,district,1.0,total_spending,11,0.433
7,district,2.0,total_spending,19,0.761
8,district,3.0,total_spending,16,0.688
9,district,4.0,total_spending,19,0.672


## 8. A simple linear probability model

We keep this intentionally simple:

`viable (0/1) = a + b × log(1 + money)`

Running a regression on a 0/1 outcome is called a **linear probability model**.
Its fitted values read as an estimated probability of being viable, which is easy
to explain — but the line can run below 0 or above 1 at the extremes, which is
impossible for a real probability. That is a known limitation of the method, not
an error in our data, and it is why we treat this as **descriptive** rather than
as a classifier.

With log money, the slope answers: how much does the estimated probability of
viability rise when a campaign roughly triples its money?


In [11]:
def viability_model(data, money, district=None):
    """Fit viability on log money, either pooled or for one district."""
    if district is None:
        pair = data[["viable_int", money]].dropna()
        scope = "all districts"
    else:
        pair = data.loc[data["district"] == district, ["viable_int", money]].dropna()
        scope = "district"

    # Needs enough candidates, and both outcomes present.
    if len(pair) < 4 or pair["viable_int"].nunique() < 2 or pair[money].nunique() < 2:
        return None

    x = np.log1p(pair[money].to_numpy(dtype=float))
    y = pair["viable_int"].to_numpy(dtype=float)

    model = sm.OLS(y, sm.add_constant(x)).fit()

    return {
        "scope": scope,
        "district": district,
        "money": money,
        "n": int(model.nobs),
        "slope": model.params[1],
        "p_value": model.pvalues[1],
        "r_squared": model.rsquared,
    }


model_rows = []

for money in ["fundraising", "total_spending"]:
    result = viability_model(analysis, money, district=None)

    if result is not None:
        model_rows.append(result)

    for district in sorted(analysis["district"].dropna().unique()):
        result = viability_model(analysis, money, district=district)

        if result is not None:
            model_rows.append(result)

viability_models = pd.DataFrame(model_rows)
display(viability_models.round({"slope": 3, "p_value": 6, "r_squared": 3}))


,scope,district,money,n,slope,p_value,r_squared
0,all districts,NaN,fundraising,65,0.220,0.000000,0.454
1,district,1.0,fundraising,13,0.275,0.000014,0.833
2,district,2.0,fundraising,20,0.296,0.001916,0.423
3,district,3.0,fundraising,15,0.292,0.000101,0.700
4,district,4.0,fundraising,17,0.149,0.024572,0.294
5,all districts,NaN,total_spending,65,0.206,0.000000,0.454
6,district,1.0,total_spending,11,0.202,0.000204,0.800
7,district,2.0,total_spending,19,0.329,0.004582,0.385
8,district,3.0,total_spending,16,0.169,0.001198,0.539
9,district,4.0,total_spending,19,0.231,0.003490,0.403


**Read the district rows with caution.** Each uses only 11 to 20 candidates, so
the R² values will bounce around. A district that looks dramatically different
may simply have few candidates. Check it in the box plots and the residual
explorer before writing anything about it.


## 9. Build the qualitative case shortlist

We take the candidates with the largest **absolute** residual in each district —
biggest misses in either direction — for fundraising and mentions.

Absolute value is the right choice here: an unexpectedly weak campaign is as
interesting as an unexpectedly strong one, and often easier to explain.

The three empty columns at the end are there for you to fill in by hand. This
table is a **research worksheet**, and once completed it becomes the backbone of
the report's qualitative section.


In [12]:
shortlist = fundraising_mentions.copy()

# Size of the miss, ignoring direction.
shortlist["absolute_residual"] = shortlist["residual"].abs()

shortlist = (
    shortlist
    .sort_values(["district", "absolute_residual"], ascending=[True, False])
    .groupby("district")
    .head(4)
)

shortlist = shortlist[[
    "district",
    "canonical_candidate",
    "is_viable",
    "fundraising",
    "total_spending",
    "mentions",
    "predicted",
    "residual",
    "absolute_residual",
]].copy()

# Which direction, in words, so the table reads without doing mental arithmetic.
shortlist["direction"] = np.where(
    shortlist["residual"] > 0, "over-performed", "under-performed"
)

# Empty columns to fill in during the qualitative pass.
shortlist["what_we_found"] = ""
shortlist["explanation_to_check"] = ""
shortlist["source_or_bio_reference"] = ""

display(shortlist.round(0))


,district,canonical_candidate,is_viable,fundraising,total_spending,mentions,predicted,residual,absolute_residual,direction,what_we_found,explanation_to_check,source_or_bio_reference
11,1,Candace Avalos,True,58894.0,NaN,22267.0,18893.0,3374.0,3374.0,over-performed,,,
8,1,Cayle Tern,False,7395.0,NaN,9061.0,11326.0,-2265.0,2265.0,under-performed,,,
0,1,Timur Ender,True,61810.0,NaN,16858.0,19069.0,-2211.0,2211.0,under-performed,,,
6,1,Doug Clove,False,1505.0,NaN,7593.0,5522.0,2071.0,2071.0,over-performed,,,
31,2,Michelle DePass,True,32578.0,NaN,32612.0,19557.0,13055.0,13055.0,over-performed,,,
19,2,Elana Pirtle-Guiney,True,47040.0,NaN,34268.0,22362.0,11906.0,11906.0,over-performed,,,
27,2,Nabil Zaghloul,False,26484.0,NaN,6482.0,17976.0,-11494.0,11494.0,under-performed,,,
25,2,Sameer Kanal,True,34712.0,NaN,31168.0,20042.0,11126.0,11126.0,over-performed,,,
45,3,Matthew (Matt) Anderson,False,710.0,NaN,4506.0,-11083.0,15589.0,15589.0,over-performed,,,
36,3,Harrison Kass,False,16058.0,NaN,9223.0,22217.0,-12994.0,12994.0,under-performed,,,


### Questions to ask about each shortlisted candidate

Work through these when reading bios and campaign coverage. They are
**hypotheses to test**, not explanations to assign.

*Could this be a data problem rather than a political one?* Check first — it is
the cheapest to rule out.

- a missing or partial finance filing;
- spending that includes independent or committee activity;
- a name-matching error in the crosswalk.

*Could it be recognition the money did not buy?*

- incumbent or former elected official;
- ran before in a previous cycle;
- existing public profile — media, activism, a known local institution.

*Could it be organisation instead of dollars?*

- union, neighbourhood association, or advocacy-group backing;
- notable endorsements;
- a large volunteer base;
- participation in public matching funds, which changes what private dollars mean.

*Could it be the ranked-choice system itself?*

- part of an informal slate whose supporters ranked them together;
- broadly acceptable as a second or third choice without being anyone's first —
  which shows up strongly in mentions but weakly in first-place votes;
- the reverse: a devoted narrow base, strong in first-place votes and weak in
  mentions.

That last pair is worth checking directly: compare a candidate's fundraising
residual for `mentions` against the one for `first_place_votes` in the explorer
above. A candidate who over-performs on one and under-performs on the other is
telling you something specific about the *shape* of their support, and that is a
genuinely interesting finding in a proportional ranked-choice election.


## 10. Findings log — exceptions and viability

**The general pattern, restated honestly**

- (Money is a strong marker of competitiveness but does not rank candidates
  perfectly. Note the R² from notebook 10 as the reminder of how much is left
  unexplained.)

**Over-performers**

- (Names, districts, and for each the hypothesis worth checking. Keep the
  hypothesis separate from the observation.)

**Under-performers**

- (Same. Note which are data-quality suspicions rather than political ones.)

**Viability**

- (How much do the viable and non-viable finance distributions overlap? Name any
  candidate who raised viable-level money and still fell short, and any who
  cleared the threshold cheaply — the second group is usually the more
  interesting.)
- (Remember: viability is derived from mentions, so this is one finding with two
  views, not two findings.)

**Mentions vs. first-place votes**

- (Any candidate whose residual flips sign between the two outcomes, and what
  that suggests about the shape of their support.)

**Data quality flags to resolve before publication**

- (Candidates with multiple spending filings, from section 1.)
- (Any candidate whose spending far exceeds their fundraising.)

**Where this leaves Question 3**

Between notebooks 10, 11, and 12 we now have the general relationship, the
fine-grained dimensions, the district variation, and a reproducible list of
candidates who depart from the pattern — with the reasons still to be
established by reading rather than by regression.


## 11. Export the case-finding outputs

In [13]:
output_dir = PROCESSED / "finance_analysis" / str(YEAR) / "question_3" / "candidate_cases"
output_dir.mkdir(parents=True, exist_ok=True)

residuals.to_csv(output_dir / "candidate_residuals.csv", index=False)
shortlist.to_csv(output_dir / "qualitative_case_shortlist.csv", index=False)
viability_summary.to_csv(output_dir / "viability_group_summary.csv", index=False)
viability_correlations.to_csv(output_dir / "viability_correlations.csv", index=False)
viability_models.to_csv(output_dir / "viability_linear_probability_models.csv", index=False)

print("SAVED:", output_dir)
print()
print("The shortlist is a worksheet: fill in the three empty columns by hand,")
print("then save it alongside the report draft.")


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/candidate_cases

The shortlist is a worksheet: fill in the three empty columns by hand,
then save it alongside the report draft.
